## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🔄 Azure AI Agent with Existing Agent - Persistent Financial Advisor 🏦

This notebook demonstrates working with pre-existing Azure AI Agents using `FoundryAgent` for production scenarios where you want to reuse a persistent Advisor agent.

## Features Covered:
- Connecting to pre-configured Azure AI Foundry agents by name
- Working with existing agents using `FoundryAgent`
- Adding function tools to existing agents
- Production patterns for reusing Financial Advisor agents

### ⚠️ Important Note ⚠️
> **Persistent agents are useful for production scenarios where you want consistent behavior across sessions. The agent retains its configuration and can be accessed by name.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Microsoft Foundry Project**: Access to a Foundry project containing an existing Prompt Agent or Hosted Agent
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `FOUNDRY_PROJECT_ENDPOINT` (legacy repository endpoint names are also supported)
   - `FOUNDRY_AGENT_NAME` (optional; defaults to `financial-services-advisor`)
4. **Dependencies**: Select the repository `.venv` kernel with `agent-framework-core==1.17.0`

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

This example connects to an existing service-managed agent by name. It does not create or delete an agent.

## Import Libraries

Import the required libraries using `FoundryAgent`:

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os
import sys
from importlib.metadata import version
from pathlib import Path
from random import uniform
from typing import Annotated

from agent_framework.foundry import FoundryAgent
from azure.identity import AzureCliCredential
from pydantic import Field

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print({package: version(package) for package in ("agent-framework-core", "agent-framework-foundry", "azure-ai-projects")})

## Initial Setup

Load environment variables from the `.env` file:

In [ ]:
from dotenv import load_dotenv

# Resolve the root from either the notebook directory or repository directory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
load_dotenv(repo_root / ".env", override=False)

endpoint = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
agent_name = os.getenv("FOUNDRY_AGENT_NAME") or os.getenv("AZURE_AI_AGENT_NAME") or "financial-services-advisor"

assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
print("Existing Foundry agent configuration loaded (values hidden)")

## Check Environment Variables

Verify that the required environment variables are set:

In [ ]:
if not endpoint:
    raise ValueError(
        "Set FOUNDRY_PROJECT_ENDPOINT, AI_FOUNDRY_PROJECT_ENDPOINT, or AZURE_AI_PROJECT_ENDPOINT in the environment."
    )
if not agent_name:
    raise ValueError("Set FOUNDRY_AGENT_NAME to an existing service-managed agent name.")

print("Project endpoint and existing agent name: configured (values hidden)")

## Define Function Tools 🏦

Let's define banking functions that our Financial Advisor agent can use:

In [ ]:
def get_account_balance(
    account_id: Annotated[str, Field(description="The customer account ID to check balance for.")],
) -> str:
    """Get the current balance for a customer account."""
    balance = round(uniform(5000, 75000), 2)
    return f"Account {account_id}: Current balance is ${balance:,.2f}"


def get_loan_rates(
    loan_type: Annotated[str, Field(description="Type of loan: mortgage, auto, personal, or business")],
) -> str:
    """Get current interest rates for different loan types."""
    rates = {
        "mortgage": {"rate": 6.5, "term": "30 years"},
        "auto": {"rate": 7.2, "term": "5 years"},
        "personal": {"rate": 10.5, "term": "3 years"},
        "business": {"rate": 8.0, "term": "10 years"}
    }
    loan_type = loan_type.lower()
    if loan_type in rates:
        return f"Current {loan_type} loan rate: {rates[loan_type]['rate']}% APR, typical term: {rates[loan_type]['term']}"
    return f"Unknown loan type. Available: mortgage, auto, personal, business"

## Connect to and Use Existing Agent 🔄

This example shows how to:
1. Connect to a pre-configured agent in Microsoft Foundry using `FoundryAgent`
2. Register local function implementations for client-side dispatch
3. Query the agent with banking-related questions

> The existing Foundry agent definition must already contain matching function-tool schemas. When `agent_name` is supplied, Agent Framework does not add or update server-side tool declarations.

**Key Pattern**: `FoundryAgent(agent_name=..., project_endpoint=..., credential=...)` connects to an existing agent by name.

In [ ]:
async def main() -> None:
    """Connect to and invoke an existing service-managed Foundry agent."""
    print("=== 🔄 Foundry Agent with Existing Agent ===")

    with AzureCliCredential() as credential:
        async with FoundryAgent(
            agent_name=agent_name,
            project_endpoint=endpoint,
            credential=credential,
            tools=[get_account_balance, get_loan_rates],
            timeout=90,
        ) as agent:
            print("✅ Connected to the configured existing Foundry agent")

            query = "What's the balance for account ACC-PROD-001 and what are the current mortgage rates?"
            print(f"\n🤔 Customer: {query}")
            async with asyncio.timeout(90):
                result = await agent.run(query)
            assert result.text, "The service returned no answer."
            print(f"🏦 Advisor: {result.text}")

## Execute the Example 🚀

Run the main function to see the existing agent workflow:

In [ ]:
# Run the main function
await main()

## Key Takeaways 📚

1. **Service-managed agent pattern**: use `FoundryAgent(agent_name=..., project_endpoint=..., credential=...)` for an existing Prompt Agent or Hosted Agent.
2. **Application-owned agent pattern**: use `Agent(client=FoundryChatClient(...))` when the application owns the model, instructions, and tools.
3. **Public APIs only**: Agent Framework 1.17.0 accepts function tools directly; no private client-layer workaround is needed.
4. **Configuration**: `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_AGENT_NAME` identify the existing agent. Legacy repository endpoint names remain supported by this notebook.
5. **Lifecycle**: context managers close the `FoundryAgent` and `AzureCliCredential`; the service call is bounded to 90 seconds.
6. **Ownership**: this notebook invokes an existing server-side agent and does not create, update, or delete it.

### Current documentation

- [Microsoft Foundry model provider](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/model-providers/microsoft-foundry)

Validated against `agent-framework-core==1.17.0`.